# 02 — Embeddings & Recherche FAISS

Ce notebook évalue :
1. Le modèle d'embeddings `paraphrase-multilingual-MiniLM-L12-v2`
2. La qualité de la recherche sémantique via FAISS (IndexFlatIP)
3. La pertinence des résultats pour des questions en français

In [ ]:
import sys, time
sys.path.insert(0, '..')

import numpy as np
from backend.services.embedder import EmbeddingService
from backend.services.chunker import TextChunker
from backend.services.pdf_extractor import PDFExtractor

## 1. Chargement du modèle d'embeddings

In [ ]:
embedder = EmbeddingService()

# Test de base
test_texts = [
    "L'intelligence artificielle est un domaine de l'informatique.",
    "Le deep learning utilise des réseaux de neurones profonds.",
    "La cuisine française est réputée dans le monde entier."
]
embeddings = embedder.embed_texts(test_texts)
print(f"Dimension des embeddings : {embeddings.shape[1]}")
print(f"Nombre d'embeddings : {embeddings.shape[0]}")
print(f"Norme L2 (doit être ~1.0) : {np.linalg.norm(embeddings[0]):.4f}")

## 2. Similarité cosinus entre phrases

In [ ]:
# Matrice de similarité
sim_matrix = np.dot(embeddings, embeddings.T)
print("Matrice de similarité :")
for i, t in enumerate(test_texts):
    print(f"\n  [{i}] {t[:50]}...")
    for j, t2 in enumerate(test_texts):
        print(f"      vs [{j}]: {sim_matrix[i][j]:.4f}")

# Vérification : IA <-> DL doit être > IA <-> cuisine
assert sim_matrix[0][1] > sim_matrix[0][2], "IA devrait être plus proche de DL que de cuisine"
print("\n✓ Vérification sémantique réussie : IA + DL > IA + cuisine")

## 3. Recherche FAISS sur un PDF de test

In [ ]:
import faiss

# Préparer des chunks depuis un PDF
PDF_PATH = "../data/sample_courses/test_course.pdf"

extractor = PDFExtractor()
result = extractor.extract(PDF_PATH)
chunker = TextChunker(chunk_size=400, overlap=50)
chunks = chunker.split(result['pages'])

# Créer les embeddings
chunk_texts = [c['text'] for c in chunks]
chunk_embeddings = embedder.embed_texts(chunk_texts)

# Construire l'index FAISS
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"Index FAISS créé : {index.ntotal} vecteurs, dimension {dim}")

In [ ]:
# Recherche sémantique
questions = [
    "Qu'est-ce que l'apprentissage supervisé ?",
    "Comment fonctionne un réseau de neurones ?",
    "Quels sont les types de données ?"
]

TOP_K = 3

for q in questions:
    q_emb = embedder.embed_query(q)
    scores, indices = index.search(q_emb.reshape(1, -1), TOP_K)
    
    print(f"\n🔍 Question : {q}")
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        chunk = chunks[idx]
        print(f"  [{rank+1}] Score: {score:.4f} | Page {chunk.get('page', '?')} | {chunk['text'][:100]}...")

## 4. Benchmark de latence

In [ ]:
# Latence d'embedding
N_RUNS = 10
times = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    embedder.embed_query("Qu'est-ce que l'intelligence artificielle ?")
    times.append(time.perf_counter() - start)

print(f"Latence embedding query ({N_RUNS} runs) :")
print(f"  Moyenne : {np.mean(times)*1000:.1f} ms")
print(f"  Médiane : {np.median(times)*1000:.1f} ms")
print(f"  P95     : {np.percentile(times, 95)*1000:.1f} ms")

# Latence recherche FAISS
q_emb = embedder.embed_query("test")
times_faiss = []
for _ in range(N_RUNS):
    start = time.perf_counter()
    index.search(q_emb.reshape(1, -1), 3)
    times_faiss.append(time.perf_counter() - start)

print(f"\nLatence FAISS search ({N_RUNS} runs) :")
print(f"  Moyenne : {np.mean(times_faiss)*1000:.2f} ms")
print(f"  Médiane : {np.median(times_faiss)*1000:.2f} ms")

## Conclusion

- Le modèle multilingue produit des embeddings de qualité pour le français
- FAISS IndexFlatIP est rapide pour des corpus de taille modérée (< 100k chunks)
- La recherche sémantique retourne des résultats pertinents par rapport à la question